<a href="https://colab.research.google.com/github/cchen744/uhi-extreme-heat-response/blob/cell-delta-uhi/notebooks/03_uhi_n_landcover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ΔUHI & Built-environment
1. **Goal**: To investigate the possible relationship between SUHI response and built environment. Since we observed a significant increase in SUHI in several cities while others not, we are assuming that if this condition-dependence might be contributed to by composition of the built environment; we test whether this is true.

2. **Outcome Variable: Defining SUHI condition-dependence**: in this step, we define SUHI's condition-dependence as:
    
    *ΔUHI = UHI_extreme − UHI_baseline*

    - UHI_extreme: SUHI when the daily mean temperature is over 90 percentile of the daily mean temperature
    - UHI_baseline: the average of SUHI when the daily mean temperature is between 50-70 percentile.

    (Percentiles are computed within the warm-season window to ensure comparability across cities.)

3. **Explanatory Variables: Built-environment characteristics**:

    **Surface composition**:
      - Impervious surface fraction
      - Vegetation / NDVI / tree cover
      - Water or bare land fraction
      - LCZ composition

    **Urban form & intensity proxies**:
      - Built-up density / road density

4. **Analytical Strategy**
  - Analytical Unit: grid cell within each city (resolution =
  - Model:
  
    *ΔUHI_cell ~ composition_cell + proxies_cell + city fixed effects*

  - Comparison logic:
    - within-city: which built factors is correlated with ΔUHI_cell
    - across-city: does this explain why some city has higher ΔUHI
  
  - Control principles:
    - Same buffer scale
    - Same spatial resolution
    - Same seasonal window



In [6]:
!git init
!git remote add origin https://github.com/cchen744/uhi-extreme-heat-response.git
!git pull origin cell-delta-uhi
!git checkout cell-delta-uhi
!git status

!git config --global user.email cchen744@wisc.edu
!git config --global user.name cchen744

Reinitialized existing Git repository in /content/.git/
error: remote origin already exists.
From https://github.com/cchen744/uhi-extreme-heat-response
 * branch            cell-delta-uhi -> FETCH_HEAD
Already up to date.
M	__pycache__/uhi_pipeline.cpython-312.pyc
Already on 'cell-delta-uhi'
Your branch is up to date with 'origin/cell-delta-uhi'.
On branch cell-delta-uhi
Your branch is up to date with 'origin/cell-delta-uhi'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   __pycache__/uhi_pipeline.cpython-312.pyc

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	drive/
	sample_data/

no changes added to commit (use "git add" and/or "git commit -a")


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
import os
import pandas as pd
import ee
import uhi_pipeline
import importlib
import geemap
from pprint import pprint

ee.Authenticate()
ee.Initialize(project='extremeweatheruhi')

DATA_DIR = Path("data/city_cell")
DATA_DIR.mkdir(parents=True, exist_ok=True)

ua_fc = ee.FeatureCollection("projects/extremeweatheruhi/assets/uac20_2025")

In [4]:
importlib.reload(uhi_pipeline)
print("uhi_pipeline module reloaded.")

uhi_pipeline module reloaded.


Before starting analysis, we need to get daily output on the level of grid cell due to the need for intra-city built environment analysis. In uhi_pipeline.py, the grid cell is defined as a 1km x 1km square grid projected from EPSG: 3875.

In [5]:
city_fc=uhi_pipeline.select_ua(ua_fc,ua_contains="Phoenix")
city_geom = city_fc.geometry() # Get city boundary
urban_region = city_geom

Map = geemap.Map()
Map.addLayer(urban_region,{},'city')
Map.centerObject(urban_region, 10)
Map

Map(center=[33.497986517451864, -112.00262242282876], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
ic = (ee.ImageCollection("MODIS/061/MYD11A1") # Extract data: MODIS Aqua Land Surface Temperature daily product
      .filterBounds(city_geom) # filtered by Phoneix' boundary
      .filterDate("2019-07-01", "2019-07-08")) # filtered by a certain time window
print("IC count:", ic.size().getInfo()) # Check how many images of the city are there within the time frame

IC count: 7


In [ ]:
ic.first().bandNames().getInfo() # Print all bands in this one MODIS product

['LST_Day_1km',
 'QC_Day',
 'Day_view_time',
 'Day_view_angle',
 'LST_Night_1km',
 'QC_Night',
 'Night_view_time',
 'Night_view_angle',
 'Emis_31',
 'Emis_32',
 'Clear_day_cov',
 'Clear_night_cov']

In [8]:
 # extract local time zone and select LCZ_Filter for built-environment analysis
lcz_img = ee.ImageCollection("RUB/RUBCLIM/LCZ/global_lcz_map/latest").first()
lcz = lcz_img.select("LCZ_Filter")

# land surface temp relevant
lst_scale_m = 1000 # modis lst resolution / sampling scale for reduceRegions()
lst_band="LST_Night_1km"
qc_band="QC_Night"

# aggregation and spatial unit parameters
agg_func="median"
unit="cell"
cell_scale_m=4000 # grid size
cell_crs="EPSG:3857"
tileScale = 1 # the bigger, the more granuality it has

# time window
start_date="2016-06-01"
end_date="2019-08-31"

# define LCZ classification rule
BUILT_MIN, BUILT_MAX = 1, 10
WATER_CODE = 17

is_built = lcz.gte(BUILT_MIN).And(lcz.lte(BUILT_MAX))
is_water = lcz.eq(WATER_CODE)
is_natural = is_built.Not().And(is_water.Not())

# define urban region and urban reference region
ring_outer_m= 12000 # buffer ring from urban area (far)
ring_inner_m= 3000 # buffer ring from urban area (near)
outer = city_geom.buffer(ring_outer_m)
inner = city_geom.buffer(ring_inner_m)
rural_region = outer.difference(inner) # rural region is defined as area 3 km to 12 km from the city boundary

urban_mask = is_built.clip(urban_region)
rural_mask = is_natural.clip(rural_region)

In [ ]:
# Beneath the urban mask I have defined, does this image contain a sufficient number of valid LST pixels for use?
ic.first().clip(urban_region).reduceRegion(reducer=ee.Reducer.count(),
            geometry=urban_region,
            scale=lst_scale_m, # lst_scale_m = 1000
            maxPixels=1e13)

In [ ]:
test_fc = uhi_pipeline.make_daily_table_cells(
    start_date, end_date,
    urban_region, rural_region,
    lst_band, qc_band,
    lst_scale_m=1000,
    cell_scale_m=4000,
    crs="EPSG:3857",
    err_m=100,
    tileScale=1)

In [ ]:
non_empty_first = test_fc.filter(ee.Filter.gt("urb_cell_n", 0))
print("Properties:", non_empty_first.first().getInfo())

Properties: {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-111.82228656719803, 33.21592806426514], [-111.78635395583326, 33.21592806426514], [-111.78635395583326, 33.24598455571292], [-111.82228656719803, 33.24598455571292], [-111.82228656719803, 33.21592806426514]]]}, 'id': '0_-3112+981', 'properties': {'LST_rur': 26.219619238476955, 'LST_urb_cell': 27.00349565217392, 'cell_id': -311099999019, 'date': '2019-07-12', 'rural_cell_n': 265, 'system:time_start': 1562889600000, 'uhi': 0.7838764136969658, 'urb_cell_n': 7}}


In [ ]:
print("total:", test_fc.size().getInfo())
print("after filterDate:", test_fc.filter(ee.Filter.gt("urb_cell_n", 0)).size().getInfo())

total: 2961
after filterDate: 1931


In [ ]:
# 2019-07-14 experienced extreme heat so I am using it as a test sample.
one_day_fc = test_fc.filter(ee.Filter.eq("date", "2016-07-14"))
print("all cells: ", one_day_fc.size().getInfo())
print("valid cells: ", one_day_fc.filter(ee.Filter.gt("urb_cell_n",0)).size().getInfo())
print(one_day_fc.filter(ee.Filter.gt("urb_cell_n",0)).limit(5).getInfo())

# Visualization
uhi_img = one_day_fc.reduceToImage(
    properties=['uhi'],
    reducer=ee.Reducer.first()
)

vis = {
    'min': -5,
    'max': 5,
    'palette': ['blue', 'white', 'red']
}

Map = geemap.Map()
Map.centerObject(urban_region, 9)
Map.addLayer(uhi_img, vis, 'UHI 2016-07-14')
Map

all cells:  423
valid cells:  41
{'type': 'FeatureCollection', 'columns': {}, 'features': [{'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-111.82228656719803, 33.18586124525484], [-111.78635395583326, 33.18586124525484], [-111.78635395583326, 33.21592806426514], [-111.82228656719803, 33.21592806426514], [-111.82228656719803, 33.18586124525484]]]}, 'id': '2_-3112+980', 'properties': {'LST_rur': 29.171156080686224, 'LST_urb_cell': 29.038205128205163, 'cell_id': -311099999020, 'cell_n': 4, 'date': '2019-07-14', 'rural_n': 1414, 'uhi': -0.13295095248106037}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-111.82228656719803, 33.21592806426514], [-111.78635395583326, 33.21592806426514], [-111.78635395583326, 33.24598455571292], [-111.82228656719803, 33.24598455571292], [-111.82228656719803, 33.21592806426514]]]}, 'id': '2_-3112+981', 'properties': {'LST_rur': 29.171156080686224, 'LST_urb_cell': 27.135345413

Map(center=[33.497986517451864, -112.00262242282876], controls=(WidgetControl(options=['position', 'transparen…

### officially start exporting data

In [10]:
# time window
start_date="2020-06-01"
end_date="2020-08-31"

# Exporting one-week data
export_fc = uhi_pipeline.make_daily_table_cells(
    start_date, end_date,
    urban_region, rural_region,
    lst_band, qc_band,
    lst_scale_m=1000,
    cell_scale_m=4000,
    crs="EPSG:3857",
    err_m=100,
    tileScale=1)

# The first feature has null LST_urb and uhi. So we use selectors to fix the output column names.
task = ee.batch.Export.table.toDrive(
    collection=export_fc,
    description=f'phoenix_daily_cell_uhi_{start_date[:7]}_{end_date[:7]}',
    fileFormat='CSV',
    folder = "EE_Exports",
    selectors=[
        'date',
        'cell_id',
        'LST_urb_cell',
        'urb_cell_n',
        'LST_rur',
        'rural_cell_n',
        'uhi',
        'system:time_start'
    ]
)
task.start()

print("Export started.")

Export started.


Check task status: https://code.earthengine.google.com/tasks

In [ ]:
commit_msg = 'UPDATE: merged cell table generation logic with city table generation logic'
! git add uhi_pipeline.py
! git commit -m commit_msg
! git push --set-upstream origin main cell-delta-uhi